In [26]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [27]:
df = pd.read_csv('../data/METABRIC_RNA_Mutation.csv')



C:\Users\DANISH LAPTOP\AppData\Local\Temp\ipykernel_14388\4103849072.py:1: DtypeWarning: Columns (0: rasgef1b_mut, 1: hras_mut, 2: smarcb1_mut, 3: siah1_mut) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/METABRIC_RNA_Mutation.csv')


In [28]:
df.head()

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,mtap_mut,ppp2cb_mut,smarcd1_mut,nras_mut,ndfip1_mut,hras_mut,prps2_mut,smarcb1_mut,stmn2_mut,siah1_mut
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0


In [29]:
target = 'overall_survival'

leakage_cols = [
    'patient_id',
    'overall_survival_months',
    'death_from_cancer'
]

df = df.drop(columns=leakage_cols)

print("Dataset Shape:", df.shape)



Dataset Shape: (1904, 690)


In [30]:
X = df.drop(columns=[target])
Y = df[target]

print("X Shape:", X.shape)
print("Y Shape:", Y.shape)

X Shape: (1904, 689)
Y Shape: (1904,)


In [31]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print("Training Data:", X_train)
print("Testing Data:", X_test)




Training Data:       age_at_diagnosis type_of_breast_surgery    cancer_type  \
1287             69.70      BREAST CONSERVING  Breast Cancer   
1602             79.20      BREAST CONSERVING  Breast Cancer   
987              72.49      BREAST CONSERVING  Breast Cancer   
1845             70.97      BREAST CONSERVING  Breast Cancer   
1833             81.14             MASTECTOMY  Breast Cancer   
...                ...                    ...            ...   
1250             72.23      BREAST CONSERVING  Breast Cancer   
600              69.48             MASTECTOMY  Breast Cancer   
923              76.06      BREAST CONSERVING  Breast Cancer   
1323             69.27      BREAST CONSERVING  Breast Cancer   
937              56.07             MASTECTOMY  Breast Cancer   

                           cancer_type_detailed cellularity  chemotherapy  \
1287  Breast Mixed Ductal and Lobular Carcinoma        High             0   
1602           Breast Invasive Ductal Carcinoma        High   

In [32]:
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

Numerical Features: 500
Categorical Features: 189


In [33]:
numerical_missing = X_train[numerical_features].isnull().sum()

numerical_missing = numerical_missing[
    numerical_missing > 0
].sort_values(ascending=False)

print("Numerical columns with missing values:")
print(numerical_missing)

Numerical columns with missing values:
tumor_stage                  406
neoplasm_histologic_grade     58
mutation_count                34
tumor_size                    13
dtype: int64


In [21]:
categorical_missing = X_train[categorical_features].isnull().sum()

categorical_missing = categorical_missing[
    categorical_missing > 0
].sort_values(ascending=False)

print("Categorical columns with missing values:")
print(categorical_missing)


Categorical columns with missing values:
3-gene_classifier_subtype         166
primary_tumor_laterality           85
cellularity                        35
er_status_measured_by_ihc          18
type_of_breast_surgery             14
cancer_type_detailed               12
tumor_other_histologic_subtype     12
oncotree_code                      12
dtype: int64


In [34]:
print(X_train['tumor_stage'].value_counts(dropna=False))

tumor_stage
2.0    620
NaN    406
1.0    391
3.0     95
4.0      9
0.0      2
Name: count, dtype: int64


In [35]:
print(X_train['tumor_stage'].dtype)

float64


In [36]:
numerical_features = numerical_features.drop('tumor_stage')
categorical_features = categorical_features.append(
    pd.Index(['tumor_stage'])
)

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))



Numerical Features: 499
Categorical Features: 190


In [37]:
numeric_imputer = SimpleImputer(strategy='median')

X_train_num = X_train[numerical_features].copy()
X_test_num = X_test[numerical_features].copy()

X_train_num_imputed = pd.DataFrame(
    numeric_imputer.fit_transform(X_train_num),
    columns=numerical_features,
    index=X_train.index
)

X_test_num_imputed = pd.DataFrame(
    numeric_imputer.transform(X_test_num),
    columns=numerical_features,
    index=X_test.index
)


In [38]:
categorical_imputer = SimpleImputer(strategy='most_frequent')

X_train_cat = X_train[categorical_features].copy()
X_test_cat = X_test[categorical_features].copy()

X_train_cat_imputed = pd.DataFrame(
    categorical_imputer.fit_transform(X_train_cat),
    columns=categorical_features,
    index=X_train.index
)

X_test_cat_imputed = pd.DataFrame(
    categorical_imputer.transform(X_test_cat),
    columns=categorical_features,
    index=X_test.index
)



In [39]:
print("Remaining missing values in numerical features:",
      X_train_num_imputed.isnull().sum().sum())

print("Remaining missing values in categorical features:",
      X_train_cat_imputed.isnull().sum().sum())



Remaining missing values in numerical features: 0
Remaining missing values in categorical features: 0


In [40]:
X_train_imputed = pd.concat([X_train_num_imputed, X_train_cat_imputed], axis=1)
X_test_imputed = pd.concat([X_test_num_imputed, X_test_cat_imputed], axis=1)

print("X_train:", X_train_imputed.shape)
print("X_test:", X_test_imputed.shape)



X_train: (1523, 689)
X_test: (381, 689)


In [41]:
missing_after = X_train_imputed.isnull().sum()
missing_after = missing_after[missing_after > 0]

print(missing_after)


Series([], dtype: int64)
